# Single-modality baseline models

### Preparation

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, cross_validate, KFold

In [2]:
df = pd.read_pickle(r"C:\Users\Juli\Documents\Master\Projekt Genomforschung\projekt_genomforschung\harmonized_data.pkl")
df.head()

,SequencingID,ModelID,TSPAN6 (7105),SCYL3 (57147),BAD (572),LAP3 (51056),SNX11 (29916),CASP10 (843),CFLAR (8837),FKBP4 (2288),...,Bit_1014,Bit_1015,Bit_1016,Bit_1017,Bit_1018,Bit_1019,Bit_1020,Bit_1021,Bit_1022,Bit_1023
0,CDS-0CqPRM,ACH-000242,7.10594,2.700053,5.803904,5.66261,4.231824,2.283468,4.735717,6.484923,...,0,0,0,0,0,0,0,0,0,0
1,CDS-0CqPRM,ACH-000242,7.10594,2.700053,5.803904,5.66261,4.231824,2.283468,4.735717,6.484923,...,0,0,0,1,0,1,0,0,0,0
2,CDS-0CqPRM,ACH-000242,7.10594,2.700053,5.803904,5.66261,4.231824,2.283468,4.735717,6.484923,...,0,0,0,0,0,0,0,0,0,0
3,CDS-0CqPRM,ACH-000242,7.10594,2.700053,5.803904,5.66261,4.231824,2.283468,4.735717,6.484923,...,0,0,0,0,1,1,0,0,0,0
4,CDS-0CqPRM,ACH-000242,7.10594,2.700053,5.803904,5.66261,4.231824,2.283468,4.735717,6.484923,...,0,0,0,1,0,1,0,0,0,0


### Defining training sets

In [ ]:
# training set for genomic features
X_genomic = df.iloc[:, 2:980].values # L1000 landmark genes (genomic features)
y_genomic = df['AUC'].values # y: drug response

# training set for single DRUG analysis
print(f"\nSingle drug testing with DrugID: {df['DRUG_ID'].value_counts().index[0]}")
df_single_drug = df[df['DRUG_ID'] == df['DRUG_ID'].value_counts().index[0]]
X_single_d = df_single_drug.iloc[:, 2:980].values
y_single_d = df_single_drug['AUC'].values

#####################################################################################
# training set for chemical features
X_chem = pd.concat([df.loc[:, df.columns.str.startswith('Bit_')], 
                    df.loc[:, ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']]], axis=1)
y_chem = df['AUC'].values

# training set for single CELL LINE analysis
print(f"Single cell line testing with ModelID: {df['ModelID'].value_counts().index[0]}")
df_single_cell = df[df['ModelID'] == df['ModelID'].value_counts().index[0]]
X_single_c = pd.concat([df_single_cell.loc[:, df.columns.str.startswith('Bit_')], 
                        df_single_cell.loc[:, ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']]], axis=1)
y_single_c = df_single_cell['AUC'].values


Single drug testing with DrugID: 1862
Single cell line testing with ModelID: ACH-000672


## Genomic features only

### Random Forest

In [ ]:
groups = df['ModelID'].values # groups: cell line identifiers (to ensure held-out validation)

# initialize the Random Forest Regressor
rf_genomic = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)

# setup GroupKFold for held-out cell line validation
gkf = GroupKFold(n_splits=5)

# perform Cross-Validation
print("Starting Cross-Validation...")
cv_results = cross_validate(
    rf_genomic, X_genomic, y_genomic, 
    groups=groups, 
    cv=gkf,
    scoring=['neg_mean_squared_error', 'r2'],
    return_train_score=True
)

# output Results
mse_scores = -cv_results['test_neg_mean_squared_error']
r2_scores = cv_results['test_r2']

print(f"Mean MSE: {np.mean(mse_scores):.4f} (+/- {np.std(mse_scores):.4f})")
print(f"Mean R2 Score: {np.mean(r2_scores):.4f}")

# feature Importance
# fit once on the full set to see which genes drive the prediction
rf_genomic.fit(X_genomic, y_genomic)
importances = pd.Series(rf_genomic.feature_importances_, index=df.iloc[:, 2:980].columns.values)
print("\nTop 5 Genetic Features:")
print(importances.sort_values(ascending=False).head(5))


Starting Cross-Validation...
Mean MSE: 0.0206 (+/- 0.0007)
Mean R2 Score: 0.0184

Top 5 Genetic Features:



Top 5 Genetic Features:
TJP1 (7082)      0.315030
SQSTM1 (8878)    0.016814
UBE3B (89910)    0.009887
GTF2E2 (2961)    0.008869
NOLC1 (9221)     0.008650
dtype: float64


### Random Forest restricted to one drug

In [73]:
# initialize a new Random Forest for this single drug
rf_single = RandomForestRegressor(
    n_estimators=100, 
    max_depth=15, 
    n_jobs=-1, 
    random_state=42
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results_single = cross_validate(
    rf_single, X_single_d, y_single_d, 
    cv=kf,
    scoring=['neg_mean_squared_error', 'r2'],
    return_train_score=True
)

print(f"Single Drug R2 Score: {np.mean(cv_results_single['test_r2']):.4f}")
rf_single.fit(X_single_d, y_single_d)
importances_single = pd.Series(rf_single.feature_importances_, index=df_single_drug.iloc[:, 2:980].columns.values)
print("\nTop 5 Genetic Features for Single Drug:")
print(importances_single.sort_values(ascending=False).head(5))

Single Drug R2 Score: 0.2134

Top 5 Genetic Features for Single Drug:
TSKU (25987)      0.029049
FBXL12 (54850)    0.023325
APP (351)         0.022270
PTK2 (5747)       0.016196
IKZF1 (10320)     0.014000
dtype: float64


## Chemical features only

In [ ]:
# get pharmacophore features as separate columns & concatenate back to the main dataframe
expanded_features = pd.DataFrame(df['PharmacophoreFeatures'].tolist())
df = pd.concat([df.drop('PharmacophoreFeatures', axis=1), expanded_features], axis=1)
df = df.fillna(0) 
print(df.head())

### Random forest

In [ ]:
X_cols = X_chem.columns.tolist()

# aggregate data: calculate mean AUC per drug
df_drug_baseline = df.groupby('DRUG_ID').agg({
    'AUC': 'mean',
    **{col: 'first' for col in X_cols}  # Chemical features are constant per DrugID
}).reset_index()

# define Features and Target
X_chem_bl = df_drug_baseline[X_cols].values.astype(float)
y_chem_bl = df_drug_baseline['AUC'].values

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

# cross-validation setup
kf = KFold(n_splits=5, shuffle=True, random_state=42)
print("Starting Cross-Validation on Aggregated Chemical Data...")
cv_results = cross_validate(
    rf_model, X_chem_bl, y_chem_bl, 
    cv=kf, 
    scoring=['r2', 'neg_mean_squared_error'],
    return_train_score=True
)

mean_r2 = np.mean(cv_results['test_r2'])
mean_mse = -np.mean(cv_results['test_neg_mean_squared_error'])

print(f"\n--- Chemical Baseline Results ---")
print(f"Mean R2 Score: {mean_r2:.4f}")
print(f"Mean MSE: {mean_mse:.4f}")

# feature Importance
rf_model.fit(X_chem_bl, y_chem_bl)
importances = pd.Series(rf_model.feature_importances_, index=X_cols)
print("\nTop 10 Chemical Drivers for General Drug Potency:")
print(importances.sort_values(ascending=False).head(10))

C:\Users\Juli\AppData\Local\Temp\ipykernel_17580\2463539562.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  }).reset_index()


Total unique drugs for baseline: 295
Starting Cross-Validation on Aggregated Chemical Data...

--- Chemical Baseline Results ---
Mean R2 Score: 0.0455
Mean MSE: 0.0111

Top 10 Chemical Drivers for General Drug Potency:
Bit_723             0.099963
Acceptor            0.050363
LumpedHydrophobe    0.049413
Bit_380             0.047848
Bit_387             0.047727
Hydrophobe          0.031315
Bit_130             0.030656
Bit_592             0.026242
Bit_802             0.024766
Bit_1004            0.023656
dtype: float64


### Random forest with single cell line
macht leider nicht so viel Sinn, weil ich ja quasi vorhersage, wie eine Zelllinie auf alle möglichen Medikamente reagiert...

In [75]:
rf_single_c = RandomForestRegressor(
    n_estimators=100, 
    max_depth=15, 
    n_jobs=-1, 
    random_state=42
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results_single = cross_validate(
    rf_single_c, X_single_c, y_single_c, 
    cv=kf,
    scoring=['neg_mean_squared_error', 'r2'],
    return_train_score=True
)

print(f"Single drug R2 score: {np.mean(cv_results_single['test_r2']):.4f}")
rf_single_c.fit(X_single_c, y_single_c)
importances_single = pd.Series(rf_single_c.feature_importances_, index= X_single_c.columns)
print("\nTop 5 drug features for chosen single drug:")
print(importances_single.sort_values(ascending=False).head(5))

Single drug R2 score: -0.0078

Top 5 drug features for chosen single drug:
Bit_924    0.014964
Bit_994    0.014943
Bit_870    0.014781
Bit_992    0.013715
Bit_853    0.012592
dtype: float64
